In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hypergraphx import Hypergraph
from hypergraphx.generation.hy_mmsbm_sampling import HyMMSBMSampler

from src.utils import hmrw_embeddings

In [ ]:
def generate_hypergraph_with_labels(seed, n_communities, n_nodes, w, avg_deg_target, max_hye_size):
    rng = np.random.default_rng(seed)
    z = rng.integers(0, n_communities, size=n_nodes)
    u = np.eye(n_communities, dtype=float)[z]

    sampler = HyMMSBMSampler(
        u=u,
        w=w,
        seed=seed,
        max_hye_size=max_hye_size,
        exact_dyadic_sampling=True,
        burn_in_steps=200,
        intermediate_steps=200,
    )
    H = next(sampler.sample(avg_deg=avg_deg_target, allow_rescaling=True))
    nodes_sorted = sorted(H.get_nodes())
    y = z[np.asarray(nodes_sorted, dtype=int)]
    return H, y

In [ ]:
def sample_hmrw_walks(H, walk_length=1000, walks_per_node=1, p=1.0, q=1.0, r=1.0, gamma=-1.0, seed=0, return_edge_walks=False):
    nodes = sorted(H.get_nodes())
    idx = {u: i for i, u in enumerate(nodes)}
    edges = [frozenset(e) for e in H.get_edges()]

    n = len(nodes)
    d = np.zeros(n)
    inc_edges = [[] for _ in range(n)]
    for e_i, e in enumerate(edges):
        for u in e:
            i = idx[u]
            d[i] += 1
            inc_edges[i].append(e_i)

    rng = np.random.default_rng(seed)
    walks = []
    edge_walks = []

    # Per-step Stage-1 selected-case counts: [same hyperedge, middle case, other case]
    case_counts = np.zeros((walk_length - 1, 3), dtype=float)
    step_counts = np.zeros(walk_length - 1, dtype=float)

    # Per-step Stage-1 possible-case counts among all candidate hyperedges.
    possible_counts = np.zeros((walk_length - 1, 3), dtype=float)
    possible_totals = np.zeros(walk_length - 1, dtype=float)

    for _ in range(walks_per_node):
        for start in rng.permutation(n):
            v_i = int(start)
            walk = [v_i]
            edge_walk = []
            prev_u_i, prev_e_i = None, None

            for t in range(walk_length - 1):
                cand_e = inc_edges[v_i]
                if not cand_e:
                    walk.append(v_i)
                    edge_walk.append(-1)
                    continue

                raw_e = np.zeros(len(cand_e))
                poss_local = np.zeros(3, dtype=float)
                for k, e2_i in enumerate(cand_e):
                    e2 = edges[e2_i]
                    if len(e2) == 0:
                        continue

                    if prev_e_i is None or prev_u_i is None:
                        a_edge = 1.0
                    else:
                        overlap_excl_v = len((edges[prev_e_i] - {nodes[v_i]}) & (e2 - {nodes[v_i]})) > 0
                        if e2_i == prev_e_i:
                            a_edge = 1.0 / p
                            poss_local[0] += 1.0
                        elif overlap_excl_v:
                            a_edge = 1.0
                            poss_local[1] += 1.0
                        else:
                            a_edge = 1.0 / q
                            poss_local[2] += 1.0

                    size_term = max(len(e2) - 1, 1)
                    raw_e[k] = a_edge * (size_term ** gamma)

                if prev_e_i is not None and prev_u_i is not None:
                    possible_counts[t] += poss_local
                    possible_totals[t] += poss_local.sum()

                s_e = raw_e.sum()
                if s_e <= 0 or not np.isfinite(s_e):
                    next_e_i = int(rng.choice(cand_e))
                else:
                    next_e_i = cand_e[int(rng.choice(len(cand_e), p=raw_e / s_e))]

                # Stage-1 transition case for diagnostics
                if prev_e_i is not None and prev_u_i is not None:
                    if next_e_i == prev_e_i:
                        case_counts[t, 0] += 1.0
                    elif len(edges[prev_e_i] & edges[next_e_i]) > 1:
                        case_counts[t, 1] += 1.0
                    else:
                        case_counts[t, 2] += 1.0
                    step_counts[t] += 1.0

                e2 = edges[next_e_i]
                cand_x = [idx[x] for x in e2 if idx[x] != v_i]
                raw_x = np.zeros(len(cand_x))
                prev_e_nodes = edges[prev_e_i] if prev_e_i is not None else None

                for k, x_i in enumerate(cand_x):
                    if d[x_i] <= 0:
                        continue
                    if prev_u_i is None:
                        a_node = 1.0
                    elif prev_e_nodes is not None and nodes[x_i] in prev_e_nodes:
                        a_node = 1.0
                    else:
                        a_node = 1.0 / r
                    raw_x[k] = a_node * (1.0 / d[x_i])

                s_x = raw_x.sum()
                x_i = cand_x[int(rng.choice(len(cand_x), p=raw_x / s_x))]
                prev_u_i, prev_e_i, v_i = v_i, next_e_i, x_i
                walk.append(v_i)
                edge_walk.append(next_e_i)

            walks.append(walk)
            edge_walks.append(edge_walk)

    case_freq = np.zeros_like(case_counts)
    valid = step_counts > 0
    case_freq[valid] = case_counts[valid] / step_counts[valid, None]

    possible_freq = np.zeros_like(possible_counts)
    valid_possible = possible_totals > 0
    possible_freq[valid_possible] = possible_counts[valid_possible] / possible_totals[valid_possible, None]

    if return_edge_walks:
        return walks, case_freq, possible_freq, edge_walks
    return walks, case_freq, possible_freq

In [ ]:
def node_coverage_over_time(walks):
    max_len = max((len(w) for w in walks), default=0)
    sums = np.zeros(max_len, dtype=float)
    counts = np.zeros(max_len, dtype=int)

    for w in walks:
        seen = set()
        for t, v in enumerate(w):
            seen.add(v)
            sums[t] += len(seen)
            counts[t] += 1

    coverage = np.zeros(max_len, dtype=float)
    mask = counts > 0
    coverage[mask] = sums[mask] / counts[mask]

    # Coverage should be non-decreasing in t.
    coverage = np.maximum.accumulate(coverage)
    return coverage


def plot_node_coverage(coverage, save_path=None):
    steps = np.arange(1, len(coverage) + 1)
    plt.figure(figsize=(6.5, 4.2))
    plt.plot(steps, coverage, marker="o", ms=3, lw=2)
    plt.xlabel("Walk Step", fontsize=11)
    plt.ylabel("Mean Unique Nodes Visited", fontsize=11)
    plt.title("Node Coverage Over Walk Steps", fontsize=12)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def hyperedge_coverage_over_time(edge_walks):
    max_len = max((len(w) for w in edge_walks), default=0)
    sums = np.zeros(max_len, dtype=float)
    counts = np.zeros(max_len, dtype=int)

    for w in edge_walks:
        seen = set()
        for t, e_i in enumerate(w):
            if e_i >= 0:
                seen.add(e_i)
            sums[t] += len(seen)
            counts[t] += 1

    cov = np.zeros(max_len, dtype=float)
    mask = counts > 0
    cov[mask] = sums[mask] / counts[mask]
    return np.maximum.accumulate(cov)

In [ ]:
def community_aware_coverage_over_time(walks, y):
    max_len = max((len(wk) for wk in walks), default=0)
    intra_sum = np.zeros(max_len, dtype=float)
    inter_sum = np.zeros(max_len, dtype=float)
    counts = np.zeros(max_len, dtype=float)

    for wk in walks:
        if len(wk) == 0:
            continue
        start_label = y[wk[0]]
        seen = set()
        for t, v in enumerate(wk):
            seen.add(v)
            intra = sum(1 for u in seen if y[u] == start_label)
            inter = len(seen) - intra
            intra_sum[t] += intra
            inter_sum[t] += inter
            counts[t] += 1.0

    intra = np.zeros(max_len, dtype=float)
    inter = np.zeros(max_len, dtype=float)
    mask = counts > 0
    intra[mask] = intra_sum[mask] / counts[mask]
    inter[mask] = inter_sum[mask] / counts[mask]

    intra = np.maximum.accumulate(intra)
    inter = np.maximum.accumulate(inter)
    return intra, inter


In [ ]:
# Generate multiple Hy-MMSBM hypergraph realizations
n_nodes = 500
n_graphs = 10
base_seed = 42


n_communities = 1
w = np.eye(n_communities, dtype=float)

avg_deg_target = 5
max_hye_size = 15

Hs_avgdeg5 = [H for H, _ in [generate_hypergraph_with_labels(
    base_seed + i, n_communities, n_nodes, w, avg_deg_target, max_hye_size
    ) for i in range(n_graphs)]]


avg_deg_target = 20
max_hye_size = 50

Hs_avgdeg20 = [H for H, _ in [generate_hypergraph_with_labels(
    base_seed + i, n_communities, n_nodes, w, avg_deg_target, max_hye_size
    ) for i in range(n_graphs)]]

n_graphs_comm = 5

avg_deg_target = 10
max_hye_size = 15

n_communities = 2
between = 0.9
within = 1 - between
w = np.array([
    [between, within],
    [within, between]
])

Hs_y_ncom_aff09 = [generate_hypergraph_with_labels(
    base_seed + i, n_communities, n_nodes, w, avg_deg_target, max_hye_size
    ) for i in range(n_graphs_comm)]

between = 0.6
within = 1 - between
w = np.array([
    [between, within],
    [within, between]
])

Hs_y_ncom_aff06 = [generate_hypergraph_with_labels(
    base_seed + i, n_communities, n_nodes, w, avg_deg_target, max_hye_size
    ) for i in range(n_graphs_comm)]

In [ ]:
### second set: communities with degree 5 and 20
n_graphs_comm = 5
n_communities = 2
base_seed = 42
n_nodes = 500



avg_deg_target = 5
max_hye_size = 15

between = 0.9
within = 1 - between
w = np.array([
    [between, within],
    [within, between]
])

Hs_y_avgdeg5_aff09 = [generate_hypergraph_with_labels(
    base_seed + i, n_communities, n_nodes, w, avg_deg_target, max_hye_size
    ) for i in range(n_graphs_comm)]

between = 0.6
within = 1 - between
w = np.array([
    [between, within],
    [within, between]
])

Hs_y_avgdeg5_aff06 = [generate_hypergraph_with_labels(
    base_seed + i, n_communities, n_nodes, w, avg_deg_target, max_hye_size
    ) for i in range(n_graphs_comm)]



avg_deg_target = 20
max_hye_size = 50


between = 0.9
within = 1 - between
w = np.array([
    [between, within],
    [within, between]
])

Hs_y_avgdeg20_aff09 = [generate_hypergraph_with_labels(
    base_seed + i, n_communities, n_nodes, w, avg_deg_target, max_hye_size
    ) for i in range(n_graphs_comm)]

between = 0.6
within = 1 - between
w = np.array([
    [between, within],
    [within, between]
])

Hs_y_avgdeg20_aff06 = [generate_hypergraph_with_labels(
    base_seed + i, n_communities, n_nodes, w, avg_deg_target, max_hye_size
    ) for i in range(n_graphs_comm)]

In [ ]:
results_5_dir = Path("results") / "walk_exploration" / f"avgdeg_5"
results_20_dir = Path("results") / "walk_exploration" / f"avgdeg_20"
results_comm_dir = Path("results") / "walk_exploration" / f"avgdeg_comm"
results_ncom_aff09_dir = Path("results") / "walk_exploration" / f"avgdeg_ncom_aff09"
results_ncom_aff06_dir = Path("results") / "walk_exploration" / f"avgdeg_ncom_aff06"

results_avgdeg5_aff09_dir = Path("results") / "walk_exploration" / f"avgdeg5_aff09"
results_avgdeg5_aff06_dir = Path("results") / "walk_exploration" / f"avgdeg5_aff06"
results_avgdeg20_aff09_dir = Path("results") / "walk_exploration" / f"avgdeg20_aff09"
results_avgdeg20_aff06_dir = Path("results") / "walk_exploration" / f"avgdeg20_aff06"

In [ ]:
# Precompute/load walks once for Hs (avgdeg_5) and reuse in all non-community plots
results_5_dir.mkdir(parents=True, exist_ok=True)

walk_cache_cov_path = results_5_dir / "walk_cache_cov.pkl"

pq_grid = [
    (0.25, 4),
    (0.5, 2),
    (1, 1),
    (2, 0.5),
    (4, 0.25)
]
walk_length = 100
walks_per_node = 5
r_fixed = 1.0
gammas_cov = [-1.0, 1.0]


if walk_cache_cov_path.exists():
    with open(walk_cache_cov_path, "rb") as f:
        walk_5_cache_cov = pickle.load(f)
    print(f"Loaded walk cache: {walk_cache_cov_path}")
else:
    Hs_cache = Hs_avgdeg5
    walk_5_cache_cov = {}
    for gamma_val in gammas_cov:
        for p_i, q_i in pq_grid:
            for g, H in enumerate(Hs_cache):
                if (gamma_val, p_i, q_i, g) in walk_5_cache_cov:
                    continue
                print((gamma_val, p_i, q_i, g))
                walks_i, case_freq, possible_freq, edge_walks_i = sample_hmrw_walks(
                    H,
                    walk_length=walk_length,
                    walks_per_node=walks_per_node,
                    p=p_i,
                    q=q_i,
                    r=r_fixed,
                    gamma=gamma_val,
                    seed=100 + g,
                    return_edge_walks=True,
                )
                walk_5_cache_cov[(gamma_val, p_i, q_i, g)] = {
                    "node": node_coverage_over_time(walks_i),
                    "hyper": hyperedge_coverage_over_time(edge_walks_i),
                    "case": case_freq,
                    "possible": possible_freq,
                }

    with open(walk_cache_cov_path, "wb") as f:
        pickle.dump(walk_5_cache_cov, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved walk cache: {walk_cache_cov_path}")

print("cached entries:", len(walk_5_cache_cov))

In [ ]:
# Precompute/load walks once for Hs (avgdeg_5) and reuse in all non-community plots
results_20_dir = Path("results") / "walk_exploration" / f"avgdeg_20"
results_20_dir.mkdir(parents=True, exist_ok=True)

walk_cache_cov_path = results_20_dir / "walk_cache_cov.pkl"

pq_grid = [
    (0.25, 4),
    (0.5, 2),
    (1, 1),
    (2, 0.5),
    (4, 0.25)
]
walk_length = 100
walks_per_node = 5
r_fixed = 1.0
gammas_cov = [-1.0, 1.0]


if walk_cache_cov_path.exists():
    with open(walk_cache_cov_path, "rb") as f:
        walk_20_cache_cov = pickle.load(f)
    print(f"Loaded walk cache: {walk_cache_cov_path}")
else:
    Hs_cache = Hs_avgdeg20
    walk_20_cache_cov = {}
    for gamma_val in gammas_cov:
        for p_i, q_i in pq_grid:
            for g, H in enumerate(Hs_cache):
                if (gamma_val, p_i, q_i, g) in walk_20_cache_cov:
                    continue
                print((gamma_val, p_i, q_i, g))
                walks_i, case_freq, possible_freq, edge_walks_i = sample_hmrw_walks(
                    H,
                    walk_length=walk_length,
                    walks_per_node=walks_per_node,
                    p=p_i,
                    q=q_i,
                    r=r_fixed,
                    gamma=gamma_val,
                    seed=100 + g,
                    return_edge_walks=True,
                )
                walk_20_cache_cov[(gamma_val, p_i, q_i, g)] = {
                    "node": node_coverage_over_time(walks_i),
                    "hyper": hyperedge_coverage_over_time(edge_walks_i),
                    "case": case_freq,
                    "possible": possible_freq,
                }

    with open(walk_cache_cov_path, "wb") as f:
        pickle.dump(walk_20_cache_cov, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved walk cache: {walk_cache_cov_path}")

print("cached entries:", len(walk_20_cache_cov))

In [ ]:
# Precompute/load walks once for Hs_y_ncom3 (community-aware plots)
results_ncom_aff09_dir.mkdir(parents=True, exist_ok=True)
walk_cache_comm_path = results_ncom_aff09_dir / "walk_cache_comm.pkl"

pq_comm_grid = [
    (0.25, 4),
    (0.5, 2),
    (1, 1),
    (2, 0.5),
    (4, 0.25),
    (0.25, 0.25),
    (4, 4)
]

walk_length_mix = 101   # steps 0..100
walks_per_node_mix = 5
gammas_mix = [-1.0]


if walk_cache_comm_path.exists():
    with open(walk_cache_comm_path, "rb") as f:
        walk_cache_comm = pickle.load(f)
    print(f"Loaded community walk cache: {walk_cache_comm_path}")
else:
    Hs_comm = Hs_y_ncom_aff09
    walk_cache_comm = {}
    for gamma_val in gammas_mix:
        for p_i, q_i in pq_comm_grid:
            for g, (H_i, y_i) in enumerate(Hs_comm):
                print((gamma_val, p_i, q_i, g))
                walks_i, _, _ = sample_hmrw_walks(
                    H_i,
                    walk_length=walk_length_mix,
                    walks_per_node=walks_per_node_mix,
                    p=p_i,
                    q=q_i,
                    gamma=gamma_val,
                    seed=500 + g,
                )
                intra_i, inter_i = community_aware_coverage_over_time(walks_i, y_i)
                walk_cache_comm[(gamma_val, p_i, q_i, g)] = {
                    "intra": intra_i,
                    "inter": inter_i,
                }

    with open(walk_cache_comm_path, "wb") as f:
        pickle.dump(walk_cache_comm, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved community walk cache: {walk_cache_comm_path}")

print("cached community entries:", len(walk_cache_comm))

In [ ]:
# Precompute/load walks once for Hs_y_ncom3 (community-aware plots)
results_ncom_aff06_dir.mkdir(parents=True, exist_ok=True)
walk_cache_comm_path = results_ncom_aff06_dir / "walk_cache_comm.pkl"

pq_comm_grid = [
    (0.25, 4),
    (0.5, 2),
    (1, 1),
    (2, 0.5),
    (4, 0.25),
    (0.25, 0.25),
    (4, 4)
]

walk_length_mix = 101   # steps 0..100
walks_per_node_mix = 5
gammas_mix = [-1.0]


if walk_cache_comm_path.exists():
    with open(walk_cache_comm_path, "rb") as f:
        walk_cache_comm = pickle.load(f)
    print(f"Loaded community walk cache: {walk_cache_comm_path}")
else:
    Hs_comm = Hs_y_ncom_aff06
    walk_cache_comm = {}
    for gamma_val in gammas_mix:
        for p_i, q_i in pq_comm_grid:
            for g, (H_i, y_i) in enumerate(Hs_comm):
                print((gamma_val, p_i, q_i, g))
                walks_i, _, _ = sample_hmrw_walks(
                    H_i,
                    walk_length=walk_length_mix,
                    walks_per_node=walks_per_node_mix,
                    p=p_i,
                    q=q_i,
                    gamma=gamma_val,
                    seed=500 + g,
                )
                intra_i, inter_i = community_aware_coverage_over_time(walks_i, y_i)
                walk_cache_comm[(gamma_val, p_i, q_i, g)] = {
                    "intra": intra_i,
                    "inter": inter_i,
                }

    with open(walk_cache_comm_path, "wb") as f:
        pickle.dump(walk_cache_comm, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved community walk cache: {walk_cache_comm_path}")

print("cached community entries:", len(walk_cache_comm))

In [ ]:
# Precompute/load walks once for Hs_y_ncom3 (community-aware plots)
results_avgdeg5_aff09_dir.mkdir(parents=True, exist_ok=True)
walk_cache_comm_path = results_avgdeg5_aff09_dir / "walk_cache_comm.pkl"

pq_comm_grid = [
    (0.25, 4),
    (1, 1),
    (4, 0.25),
]

walk_length_mix = 101   # steps 0..100
walks_per_node_mix = 5
gammas_mix = [-1.0]


if walk_cache_comm_path.exists():
    with open(walk_cache_comm_path, "rb") as f:
        walk_cache_comm = pickle.load(f)
    print(f"Loaded community walk cache: {walk_cache_comm_path}")
else:
    Hs_comm = Hs_y_avgdeg5_aff09
    walk_cache_comm = {}
    for gamma_val in gammas_mix:
        for p_i, q_i in pq_comm_grid:
            for g, (H_i, y_i) in enumerate(Hs_comm):
                print((gamma_val, p_i, q_i, g))
                walks_i, _, _ = sample_hmrw_walks(
                    H_i,
                    walk_length=walk_length_mix,
                    walks_per_node=walks_per_node_mix,
                    p=p_i,
                    q=q_i,
                    gamma=gamma_val,
                    seed=500 + g,
                )
                intra_i, inter_i = community_aware_coverage_over_time(walks_i, y_i)
                walk_cache_comm[(gamma_val, p_i, q_i, g)] = {
                    "intra": intra_i,
                    "inter": inter_i,
                }

    with open(walk_cache_comm_path, "wb") as f:
        pickle.dump(walk_cache_comm, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved community walk cache: {walk_cache_comm_path}")

print("cached community entries:", len(walk_cache_comm))

In [ ]:
# Precompute/load walks once for Hs_y_ncom3 (community-aware plots)
results_avgdeg5_aff06_dir.mkdir(parents=True, exist_ok=True)
walk_cache_comm_path = results_avgdeg5_aff06_dir / "walk_cache_comm.pkl"

pq_comm_grid = [
    (0.25, 4),
    (1, 1),
    (4, 0.25),
]

walk_length_mix = 101   # steps 0..100
walks_per_node_mix = 5
gammas_mix = [-1.0]


if walk_cache_comm_path.exists():
    with open(walk_cache_comm_path, "rb") as f:
        walk_cache_comm = pickle.load(f)
    print(f"Loaded community walk cache: {walk_cache_comm_path}")
else:
    Hs_comm = Hs_y_avgdeg5_aff06
    walk_cache_comm = {}
    for gamma_val in gammas_mix:
        for p_i, q_i in pq_comm_grid:
            for g, (H_i, y_i) in enumerate(Hs_comm):
                print((gamma_val, p_i, q_i, g))
                walks_i, _, _ = sample_hmrw_walks(
                    H_i,
                    walk_length=walk_length_mix,
                    walks_per_node=walks_per_node_mix,
                    p=p_i,
                    q=q_i,
                    gamma=gamma_val,
                    seed=500 + g,
                )
                intra_i, inter_i = community_aware_coverage_over_time(walks_i, y_i)
                walk_cache_comm[(gamma_val, p_i, q_i, g)] = {
                    "intra": intra_i,
                    "inter": inter_i,
                }

    with open(walk_cache_comm_path, "wb") as f:
        pickle.dump(walk_cache_comm, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved community walk cache: {walk_cache_comm_path}")

print("cached community entries:", len(walk_cache_comm))

In [ ]:
# Precompute/load walks once for Hs_y_ncom3 (community-aware plots)
results_avgdeg20_aff09_dir.mkdir(parents=True, exist_ok=True)
walk_cache_comm_path = results_avgdeg20_aff09_dir / "walk_cache_comm.pkl"

pq_comm_grid = [
    (0.25, 4),
    (1, 1),
    (4, 0.25),
]

walk_length_mix = 101   # steps 0..100
walks_per_node_mix = 5
gammas_mix = [-1.0]


if walk_cache_comm_path.exists():
    with open(walk_cache_comm_path, "rb") as f:
        walk_cache_comm = pickle.load(f)
    print(f"Loaded community walk cache: {walk_cache_comm_path}")
else:
    Hs_comm = Hs_y_avgdeg20_aff09
    walk_cache_comm = {}
    for gamma_val in gammas_mix:
        for p_i, q_i in pq_comm_grid:
            for g, (H_i, y_i) in enumerate(Hs_comm):
                print((gamma_val, p_i, q_i, g))
                walks_i, _, _ = sample_hmrw_walks(
                    H_i,
                    walk_length=walk_length_mix,
                    walks_per_node=walks_per_node_mix,
                    p=p_i,
                    q=q_i,
                    gamma=gamma_val,
                    seed=500 + g,
                )
                intra_i, inter_i = community_aware_coverage_over_time(walks_i, y_i)
                walk_cache_comm[(gamma_val, p_i, q_i, g)] = {
                    "intra": intra_i,
                    "inter": inter_i,
                }

    with open(walk_cache_comm_path, "wb") as f:
        pickle.dump(walk_cache_comm, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved community walk cache: {walk_cache_comm_path}")

print("cached community entries:", len(walk_cache_comm))

In [ ]:
# Precompute/load walks once for Hs_y_ncom3 (community-aware plots)
results_avgdeg20_aff06_dir.mkdir(parents=True, exist_ok=True)
walk_cache_comm_path = results_avgdeg20_aff06_dir / "walk_cache_comm.pkl"

pq_comm_grid = [
    (0.25, 4),
    (1, 1),
    (4, 0.25),
]

walk_length_mix = 101   # steps 0..100
walks_per_node_mix = 5
gammas_mix = [-1.0]


if walk_cache_comm_path.exists():
    with open(walk_cache_comm_path, "rb") as f:
        walk_cache_comm = pickle.load(f)
    print(f"Loaded community walk cache: {walk_cache_comm_path}")
else:
    Hs_comm = Hs_y_avgdeg20_aff06
    walk_cache_comm = {}
    for gamma_val in gammas_mix:
        for p_i, q_i in pq_comm_grid:
            for g, (H_i, y_i) in enumerate(Hs_comm):
                print((gamma_val, p_i, q_i, g))
                walks_i, _, _ = sample_hmrw_walks(
                    H_i,
                    walk_length=walk_length_mix,
                    walks_per_node=walks_per_node_mix,
                    p=p_i,
                    q=q_i,
                    gamma=gamma_val,
                    seed=500 + g,
                )
                intra_i, inter_i = community_aware_coverage_over_time(walks_i, y_i)
                walk_cache_comm[(gamma_val, p_i, q_i, g)] = {
                    "intra": intra_i,
                    "inter": inter_i,
                }

    with open(walk_cache_comm_path, "wb") as f:
        pickle.dump(walk_cache_comm, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved community walk cache: {walk_cache_comm_path}")

print("cached community entries:", len(walk_cache_comm))